# 跨文献 Suzuki 实训

4篇、40条选定结果。默认使用真实响应回放，API调用为0。详细分步手册见 docs/BATCH_MANUAL.md；参考仍需教师审校。

In [ ]:
from pathlib import Path
import os
from reaction_knowledge_lab.files import ROOT, read_json
from reaction_knowledge_lab.batch import batch_demo, batch_verify, batch_report
from reaction_knowledge_lab.store import get_history, query_records
base = Path(os.environ.get('RKL_NOTEBOOK_RUN_DIR', str(ROOT / 'runs/notebook-batch')))
work = base / 'batch'
result = batch_verify(work) if (work / 'knowledge.sqlite').exists() else batch_demo(work)
assert result['passed']
result['checks']

## 从真实模型结果到可审查记录

先阅读 raw 和 aligned，再看参考差异。差异中有表述差异，不能直接当模型错误率。

In [ ]:
raw = read_json(work / 'raw.json')['records']
print('候选条数:', len(raw))
raw[0]

In [ ]:
read_json(work / 'aligned.alignment.json')

## 共同转化不等于同一次实验

对溴甲苯＋苯硼酸在三篇论文中出现，记录各自保留。查找同产物但原料不同的反例 f6-g。

In [ ]:
db = work / 'knowledge.sqlite'
groups = batch_report(db)['cross_source_transformations']
[g for g in groups if 'nc2019-t1-9' in g['record_ids']]

In [ ]:
rows = query_records(db, name='4-methylbiphenyl')
[(r['record_id'], [c['name'] for c in r['inputs'] if c['role']=='reactant'], r['product']['yield_percent']) for r in rows]

## 修订和增量更新

27%是明确注入的教学错误，不是模型真实错误。观察历史及29→28→29查询回归，再检查增量导入幂等。

In [ ]:
[(h['revision'],h['record']['product']['yield_percent'],h['reason']) for h in get_history(db,'nc2019-t1-9')]

In [ ]:
read_json(work / 'query-regression.json'), read_json(work / 'incremental.json')

## 不能被软件自动消除的科学疑点

对照NC2025正文Fig.3与SI第50页，两个产率冲突保留双方说法。JACS产物8是五次合并结果。请使用docs/REVIEW_RETURN.md记录判断。

In [ ]:
report = batch_report(db)
report['source_conflict_ids'], report['pooled_result_ids']

In [ ]:
validation = read_json(work / 'validation/validation.json')
{'ORD errors':len(validation['ord']['dataset_errors']), 'ORD warnings':len(validation['ord']['dataset_warnings']), 'entity rules':validation['rules']['passed']}

## 作业

提交两组跨论文关联、一个不能合并的反例、一条修订历史、对合并结果与来源冲突的解释。引用实际图/页/脚注，注明使用了免费回放。